<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_reviews_asyncio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [1]:
# @title Dependencies

# Installation
!pip install pandas pyarrow -q
!pip install numpy -q
!pip install tqdm -q
!pip install openai -q
!pip install groq -q

# First-party installations
import itertools
import re
import json
import hashlib
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os
import asyncio

# Third-party installations
from google.colab import drive
import google.colab.userdata
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import AsyncOpenAI
from openai import APIError
from groq import Groq
from google import genai
from google.genai import types

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.6 MB/s eta 0:00:00


In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content ### ADJUSTED FOR TEST PURPOSE!
DATASET_DIR = os.path.join(WORKING_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_results.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_results.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.


In [15]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = False # SIMULATION_ACTIVE = False activates the API calls

# Define the number of reviews to generate per paper-parameter combo
NUM_ITERATIONS_PER_PAPER_PER_COMBO = 2

# Define the number of paper-parameter combos to process in parallel
NUM_PARALLEL_BATCH_SIZE = 10

# Subsetting paper content
PAPER_SUBSETTING_ACTIVE = True # PAPER_SUBSETTING_ACTIVE = True activates the subsetting
NUM_SUBSET = 10 # Number of papers per group

# Subsetting configurations
CONFIG_SUBSETTING_ACTIVE = True # CONFIG_SUBSETTING_ACTIVE = True activates the subsetting
CONFIG_SUBSET_MODELS = [
    "gpt-5-mini-2025-08-07"
    ] # Models to include in subset test
CONFIG_NUM_SUBSET_PER_MODEL = 2 # Number of parameter combinations to keep per model after filtering

# Probability for simulating errors (notes or review generation)
SIMULATION_ERROR_PROBABILITY = 0.1

### --- Parameters --- ###

# OpenAI
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-11-20", # Changed to a newer model version
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Groq
GROQ_MODELS = [
    "llama-3.3-70b-versatile"
]

# Gemini
GEMINI_MODELS = [
    "gemini-3-flash-preview" # new model
]

# Models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07",
    "gemini-3-flash-preview"
}

# Define temperature parameters (as used by Cummins 2025)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins 2025)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "Explain your thought process step by step."
]

# Define note taking (adopted from Robertson and Kayejo (2025))
NOTE_TAKING = [
    "Faithful",
    "Generic", ### ADOPTED FOR TEST PURPOSES!
    "Objective"
]

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    note_taking: Literal[*NOTE_TAKING]

### --- Content --- ####

# Target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Note taking strategies/instructions, adopted from Robertson and Koyejo (2025)
NOTE_TAKING_FAITHFUL = f"Take detailed, accurate notes on the paper for an ICLR style review. Focus on precisely capturing the actual methods, results, and contributions without any distortion. Just output the notes."
NOTE_TAKING_OBJECTIVE = f"Take comprehensive notes on the paper's technical content, experimental design, and results. Document both strengths and limitations objectively. Just output the notes."
NOTE_TAKING_EVALUATION = f"Take meticulous notes covering all aspects of the paper including theoretical foundations, experimental methodology, results, and implications. Note specific equations, theorem numbers, and experimental details. Just output the notes."
NOTE_TAKING_GENERIC = f"Without reading the paper, generate generic notes for an ICLR review that could apply to almost any machine learning paper. Use vague terminology and avoid any specific details about methods, experiments, or results. Just output these generic notes."
NOTE_TAKING_PROMPTS = {
    "Faithful": NOTE_TAKING_FAITHFUL,
    "Generic": NOTE_TAKING_GENERIC, ###! ADOPTED FOR TEST PURPOSES
    "Objective": NOTE_TAKING_OBJECTIVE

}

# Review generation instruction, adopted from Robertson and Koyejo (2025)
REVIEW_GENERATION = f"""

Create an ICLR-style review following this specific structure:

# Summary Of The Paper
Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses
Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility
Evaluate based on your notes.

# Summary Of The Review
Provide a 2-3 sentence distillation of your overall assessment.

# Correctness
Rate on a scale of 1-5.

# Technical Novelty And Significance
Rate on a scale of 1-5.

# Empirical Novelty And Significance
Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

"""

# Predefined placeholder strings for consistent error/status reporting
NOTE_TAKING_FAILURE = "ERROR: NOTE TAKING NOT SUCCESSFUL"
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE = "ERROR: UNKNOWN STATE REACHED"
UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING = "ERROR: UNKNOWN STATE REACHED IN PARALLEL PROCESSING"

### --- API/Client --- ###

MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random
SEED_VALUE = 45 # Random
MAX_TOKENS = 2000 # As Robertson and Kayejo (2025) (in their repo)
MAX_TOKENS_INSTRUCTION = "Please stick to an output of maximum 2000 tokens."

# OpenAI API-key
OPENAI_KEY = google.colab.userdata.get('OPENAI_API_KEY')

# Groq API-key
GROQ_KEY = google.colab.userdata.get('GROQ_API_KEY')

# Gemini API-key
GEMINI_API_KEY = google.colab.userdata.get('GEMINI_API_KEY')

# Groq URL
GROQ_URL ="https://api.groq.com/openai/v1"

In [18]:
# @title Data import

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_results_authors.parquet")) ### ADJUSTED FOR TEST PURPOSES!

# Select target columns
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']].copy()

# Remove the HEADER_TO_REMOVE string from the 'parsed_text' column
paper_content['parsed_text'] = paper_content['parsed_text'].str.replace(HEADER_TO_REMOVE, '', regex=False).str.strip()

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

,paper_id,abstract,parsed_text
0,author_0,Abstract placeholder for authors dataset.,Under review as a conference paper at ICLR 202...
1,author_1,Abstract placeholder for authors dataset.,A THEORY OF REPRESENTATION LEARNING IN NEURAL ...
2,author_2,Abstract placeholder for authors dataset.,Under review as a conference paper at ICLR 202...


(500, 3)

In [19]:
# @title Subset paper content table (if active)

if PAPER_SUBSETTING_ACTIVE:
    paper_content = paper_content.head(NUM_SUBSET)
    print(f"Paper content subsetted to {NUM_SUBSET} rows.")
    display(paper_content.head(3))
    display(paper_content.shape)

    # Update targets after subsetting
    targets = paper_content.to_dict('records')
else:
    print("Paper content subsetting is inactive.")

Paper content subsetted to 10 rows.


,paper_id,abstract,parsed_text
0,author_0,Abstract placeholder for authors dataset.,Under review as a conference paper at ICLR 202...
1,author_1,Abstract placeholder for authors dataset.,A THEORY OF REPRESENTATION LEARNING IN NEURAL ...
2,author_2,Abstract placeholder for authors dataset.,Under review as a conference paper at ICLR 202...


(10, 3)

### Client

In [20]:
class AsyncLLMClient:
    def __init__(self, seed: int, simulate: bool = True):
        """Initialize AsyncLLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)

        # Set placeholders for real clients when simulate=False:
        self._openai = None
        self._groq_openai_client = None
        self._gemini_client = None

    def _maybe_init_clients(self):
        """Start simulation/API clients"""
        if self.simulate:
            return
        # Initialize OpenAI client (AsyncOpenAI)
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = AsyncOpenAI(api_key=OPENAI_KEY)

        # Initialize Groq client using OpenAI client interface (AsyncOpenAI)
        if self._groq_openai_client is None and "GROQ_KEY" in globals():
            self._groq_openai_client = AsyncOpenAI(api_key=GROQ_KEY, base_url=GROQ_URL)

        # Configure Gemini API globally and instantiate client
        if self._gemini_client is None and "GEMINI_API_KEY" in globals():
            self._gemini_client = genai.Client(api_key=GEMINI_API_KEY)

    async def call(
        self,
        prompt_content: str,
        model: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        chain_of_thought: str,
        note_taking: str,
        max_retries: float = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        max_tokens: int = MAX_TOKENS,
    ) -> Tuple[str, str]:
        """
        Defines the API calls / simulations according to the configuration setup.
        Returns: notes, review.
        """

        self._maybe_init_clients()

        # --- Step 1: Generate notes ---

        # Construct final prompt basis
        note_taking_instruction = NOTE_TAKING_PROMPTS.get(note_taking, note_taking)
        final_note_taking_prompt = f"{note_taking_instruction}\n\n{prompt_content}{note_taking_instruction}\n\n".strip()

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_note_taking_prompt += f" {chain_of_thought}"

        # Add maximum token instruction for reasoning models only
        if model in REASONING_MODELS:
            final_note_taking_prompt += f" {MAX_TOKENS_INSTRUCTION}"

        # Set provider for note-taking model
        provider_notes = "openai"
        if model in GROQ_MODELS:
            provider_notes = "groq"
        elif model in GEMINI_MODELS:
            provider_notes = "gemini"

        # Set temporary placeholder for the notes
        generated_notes = None

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial note
                if self.simulate:
                    if self.rng.uniform(0, 1) < SIMULATION_ERROR_PROBABILITY:
                        generated_notes = NOTE_TAKING_FAILURE
                    else:
                        generated_notes = (
                            f"This is a simulated note based on: Model='{model}', Temp='{temperature}', "
                            f"Effort='{reasoning_effort}', CoT='{chain_of_thought}', Notes='{note_taking}'. "
                        )
                    break
                # Call API
                else:
                    # For OpenAI (reasoning)
                    if provider_notes == "openai" and model in REASONING_MODELS:
                        kwargs = {}
                        # Set hyperparameter
                        kwargs["reasoning"] = {"effort": reasoning_effort}

                        # Specify the user message
                        msgs = [{"role": "user", "content": final_note_taking_prompt}]

                        # Specify the client to use
                        client_to_use = self._openai

                        # Call client and return the notes
                        resp = await client_to_use.responses.create(
                            model=model,
                            input=msgs,
                            **kwargs,
                        )
                        # Extract the notes
                        generated_notes = resp.output_text
                        break

                    # For OpenAI (non-reasoning) and Groq
                    if (provider_notes == "openai" and model not in REASONING_MODELS) or provider_notes == "groq":
                        kwargs = {}
                        # Set hyperparameter
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # Specify the user message
                        msgs = [{"role": "user", "content": final_note_taking_prompt}]

                        # Specify the client to use
                        client_to_use = self._groq_openai_client if provider_notes == "groq" else self._openai

                        # Call client and return the notes
                        resp = await client_to_use.chat.completions.create(
                            model=model,
                            messages=msgs,
                            max_tokens=max_tokens,
                            **kwargs,
                        )
                        # Extract the notes
                        generated_notes = resp.choices[0].message.content or ""
                        break

                    # For Gemini
                    elif provider_notes == "gemini":
                        # Set hyperparameter
                        gen_content_config_args = {}
                        gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                        # Call client and return the notes (wrapped in to_thread for async compatibility)
                        resp = await asyncio.to_thread(
                            self._gemini_client.models.generate_content,
                            model=model,
                            contents=final_note_taking_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        # Extract the notes
                        generated_notes = resp.text
                        break

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Note Taking] provider={provider_notes} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    # Return failure string across both outputs
                    return NOTE_TAKING_FAILURE, REVIEW_GENERATION_FAILURE
                await asyncio.sleep(retry_delay)

        # Return failure string across both outputs
        if generated_notes is None or generated_notes == NOTE_TAKING_FAILURE:
            return NOTE_TAKING_FAILURE, REVIEW_GENERATION_FAILURE

        # --- Step 2: Generate the main review ---

        # Construct final prompt basis
        final_review_prompt = f"{REVIEW_GENERATION}\n\n{generated_notes}{REVIEW_GENERATION}\n\n".strip()

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_review_prompt += f" {chain_of_thought}"

        # Add maximum token instruction for reasoning models only
        if model in REASONING_MODELS:
            final_review_prompt += f" {MAX_TOKENS_INSTRUCTION}"

        # Set provider for review model
        provider_review = "openai"
        if model in GROQ_MODELS:
            provider_review = "groq"
        elif model in GEMINI_MODELS:
            provider_review = "gemini"

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial review
                if self.simulate:
                    if self.rng.uniform(0, 1) < SIMULATION_ERROR_PROBABILITY:
                        generated_review = self.rng.choice([REVIEW_GENERATION_FAILURE, UNKNOWN_ERROR_STATE])
                    else:
                        generated_review = (
                            f"This is a simulated review: Model='{model}', Temp={temperature}, "
                            f"Effort='{reasoning_effort}', chain_of_thought='{chain_of_thought}'. "
                        )
                    return generated_notes, generated_review
                # Call API
                else:
                    # For OpenAI (reasoning)
                    if provider_review == "openai" and model in REASONING_MODELS:
                        kwargs = {}
                        # Set hyperparameter
                        kwargs["reasoning"] = {"effort": reasoning_effort}

                        # Specify the user message
                        msgs = [{"role": "user", "content": final_review_prompt}]

                        # Specify the client to use
                        client_to_use = self._openai

                        # Call client and return the notes
                        resp = await client_to_use.responses.create(
                            model=model,
                            input=msgs,
                            **kwargs,
                        )
                        # Extract the notes
                        generated_review = resp.output_text

                    # For OpenAI (non-reasoning) and Groq
                    elif (provider_review == "openai" and model not in REASONING_MODELS) or provider_review == "groq":
                        kwargs = {}
                        # Set hyperparameter
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # Specify the user message
                        msgs = [{"role": "user", "content": final_review_prompt}]

                        # Specify the client to use
                        client_to_use = self._groq_openai_client if provider_review == "groq" else self._openai

                        # Call client and return the notes
                        resp = await client_to_use.chat.completions.create(
                            model=model,
                            messages=msgs,
                            max_tokens=max_tokens,
                            **kwargs,
                        )
                        # Extract the notes
                        generated_review = resp.choices[0].message.content or ""

                    # For Gemini
                    elif provider_review == "gemini":

                        # Set and populate hyperparameter
                        gen_content_config_args = {}
                        gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                        # Call client and return the review (wrapped in to_thread for async compatibility)
                        resp = await asyncio.to_thread(
                            self._gemini_client.models.generate_content,
                            model=model,
                            contents=final_review_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        # Extract the review
                        generated_review = resp.text

                    return generated_notes, generated_review

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Review Generation] provider={provider_review} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    # Return notes and failure message for review generation
                    return generated_notes, REVIEW_GENERATION_FAILURE
                await asyncio.sleep(retry_delay)

        # Return notes and unknown error for review generation
        return generated_notes, UNKNOWN_ERROR_STATE

### Helper for output messages

In [21]:
def _fmt_combo(combo: ParamCombo) -> str:
    """Helper to format the ParamCombo into a human-readable string"""
    parts = []
    parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None:
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    parts.append(f"note_taking={combo.note_taking}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    """Helper to enrich the ParamCombo string with information for output messages"""
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

### Simulation definition

In [22]:
async def run_parameter_combo(
    combo: ParamCombo,
    review_target: Dict,
    client: Optional['AsyncLLMClient'] = None,
) -> pd.DataFrame:
    """
    Sets id, content and parameters for the client call.
    Calls helper function to print the information.
    Calls the client and returns the results in single data frame row.
    """
    # If client specified then use client, otherwise fake client for test
    client = client or AsyncLLMClient(simulate=True)

    # Extract the current content
    paper_content = review_target['parsed_text']

    # Extract the current id
    doc_id = review_target["paper_id"]

    # Initialize a dictionary for the current id and parameters
    current_paper_record = {
        "paper_id": doc_id,
        "model": combo.model,
        "temperature": combo.temperature,
        "reasoning_effort": combo.reasoning_effort,
        "chain_of_thought": combo.chain_of_thought,
        "note_taking": combo.note_taking,
        "run_signature": RUN_SIGNATURE
    }

    # Loop to generate multiple notes/reviews sequentially for a single paper-combo
    for review_idx in range(NUM_ITERATIONS_PER_PAPER_PER_COMBO):

        # Include informative output message
        log_call(doc_id, combo, review_number=review_idx + 1)

        # Parse current content and parameters to the client
        notes, review = await client.call(
            model=combo.model,
            prompt_content=paper_content,
            temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
            reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
            chain_of_thought=combo.chain_of_thought,
            note_taking=combo.note_taking
        )

        # Store notes
        current_paper_record[f"llm_notes_{review_idx + 1}"] = notes
        # Store review
        current_paper_record[f"llm_review_{review_idx + 1}"] = review

    # Convert the record into a DataFrame row
    df = pd.DataFrame(current_paper_record, index=[0])
    return df

### Configuration grid

In [23]:
# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS + GEMINI_MODELS

# Generate every single combination, with NOTE_TAKING varying last (fastest)
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, NOTE_TAKING))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")

display(experiment_config)

Total valid parameter combos: 156


,model,temperature,reasoning_effort,chain_of_thought,note_taking
0,gpt-5-mini-2025-08-07,NaN,low,,Faithful
1,gpt-5-mini-2025-08-07,NaN,low,,Generic
2,gpt-5-mini-2025-08-07,NaN,low,,Objective
3,gpt-5-mini-2025-08-07,NaN,low,Explain your thought process step by step.,Faithful
4,gpt-5-mini-2025-08-07,NaN,low,Explain your thought process step by step.,Generic
...,...,...,...,...,...
151,gemini-3-flash-preview,NaN,high,,Generic
152,gemini-3-flash-preview,NaN,high,,Objective
153,gemini-3-flash-preview,NaN,high,Explain your thought process step by step.,Faithful
154,gemini-3-flash-preview,NaN,high,Explain your thought process step by step.,Generic


In [24]:
# @title Subset configuration grid (if active)

if CONFIG_SUBSETTING_ACTIVE:
    print(f"Applying configuration subsetting...")
    original_num_configs = len(experiment_config)

    # Filter by specific models if CONFIG_SUBSET_MODELS is not empty
    if CONFIG_SUBSET_MODELS:
        experiment_config = experiment_config[experiment_config['model'].isin(CONFIG_SUBSET_MODELS)]
        print(f"Configuration grid filtered to models: {CONFIG_SUBSET_MODELS}. Number of configs: {len(experiment_config)}")

    # Group by model and take the first CONFIG_NUM_SUBSET_PER_MODEL rows from each group
    if CONFIG_NUM_SUBSET_PER_MODEL > 0:
        experiment_config = experiment_config.groupby('model').head(CONFIG_NUM_SUBSET_PER_MODEL).reset_index(drop=True)
        print(f"Configuration grid further subsetted to {CONFIG_NUM_SUBSET_PER_MODEL} rows per model.")

    print(f"Original total valid parameter combos: {original_num_configs}")
    print(f"Final total valid parameter combos after subsetting: {len(experiment_config)}")

    display(experiment_config)
else:
    print("Configuration grid subsetting is inactive.")

Applying configuration subsetting...
Configuration grid filtered to models: ['gpt-5-mini-2025-08-07']. Number of configs: 12
Configuration grid further subsetted to 2 rows per model.
Original total valid parameter combos: 156
Final total valid parameter combos after subsetting: 2


,model,temperature,reasoning_effort,chain_of_thought,note_taking
0,gpt-5-mini-2025-08-07,NaN,low,,Faithful
1,gpt-5-mini-2025-08-07,NaN,low,,Generic


### Helper for logging

In [25]:
def _nan_to_none(x):
    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x


RUN_SIGNATURE = hashlib.sha1(
    json.dumps(
        {
            "note_taking_prompts": NOTE_TAKING_PROMPTS,
            "review_generation_prompt": REVIEW_GENERATION,
            "max_tokens": MAX_TOKENS,
            "max_tokens_instruction": MAX_TOKENS_INSTRUCTION,
            "num_reviews_per_paper": NUM_ITERATIONS_PER_PAPER_PER_COMBO,
        },
        sort_keys=True,
        ensure_ascii=True,
    ).encode("utf-8")
).hexdigest()[:12]


def combo_key_tuple(row) -> tuple:
    """Combines run and configuration settings with None instead of NaN"""
    run_signature = row["run_signature"] if "run_signature" in row and pd.notna(row["run_signature"]) else "__legacy__"
    return (
        run_signature,
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["note_taking"]
    )

def combo_key_str(row) -> str:
    """Creates readable combo string for logging"""
    t = combo_key_tuple(row)
    return f"run={t[0]}|model={t[1]}|temp={t[2]}|re={t[3]}|cot={t[4]}|notes={t[5]}"

### LLM review generation

In [26]:
if __name__ == '__main__':
    async def main():

        print(f"Processing and saving intermediate results to: {RESULTS_JSON_PATH}", flush = True)
        print(f"Logging successful iterations to: {LOG_JSON_PATH}", flush = True)

        # Initialize client
        client = AsyncLLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    ### --- 1. Check for already existing logs and results --- ###

        # Initialize keys
        done_keys = set()

        # Helper to extract relevant columns for combo key creation
        combo_cols = ["run_signature", "model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

        # Check and read successful paper-parameter combos from JSON log file
        if os.path.exists(LOG_JSON_PATH):
            try:
                with open(LOG_JSON_PATH, 'r') as f:
                    for line in f:
                        entry = json.loads(line)
                        if 'paper_id' in entry and 'completed_param_combo' in entry:
                            done_keys.add((entry['paper_id'], entry['completed_param_combo']))
                print(f"Loaded {len(done_keys)} completed paper-parameter combo entries from JSON log.")
            except Exception as e:
                print(f"Error reading existing JSON log file: {e}. Starting with empty log.")
                done_keys = set()

        # Check and read successful paper-parameter combos from JSON result file
        if os.path.exists(RESULTS_JSON_PATH):
            try:
                with open(RESULTS_JSON_PATH, 'r') as f:
                    for line in f:
                        entry = json.loads(line)
                        entry_for_combo_key = entry.copy()
                        paper_id = entry["paper_id"]
                        param_combo_str_current = combo_key_str(entry_for_combo_key)
                        done_keys.add((paper_id, param_combo_str_current))

                print(f"Loaded {len(done_keys)} completed paper-parameter combo entries from JSONL results.")

            except Exception as e:
                print(f"Error reading existing JSONL results file: {e}. Moving on.")

        print(f"Total already-completed paper-configuration combos: {len(done_keys)}")

    ### --- 2. Create notes and reviews for all paper-parameter combos --- ###

        all_tasks_to_run = []
        task_metadata = []
        total_combos = len(experiment_config) * len(targets)
        processed_combos = 0

        # Prepare all unique (combo, paper) pairs that need to be processed
        for idx_combo, row_combo in experiment_config.iterrows():
            for idx_paper, paper_data in enumerate(targets):
                paper_id = paper_data["paper_id"]

                # Set string representation of the current combo
                row_combo_for_key = row_combo.copy()
                row_combo_for_key["run_signature"] = RUN_SIGNATURE
                param_combo_str_current = combo_key_str(row_combo_for_key)
                paper_combo_key = (paper_id, param_combo_str_current)

                # Check if this specific paper-combo is already completed
                if paper_combo_key in done_keys:
                    print(f"[SKIP] Paper {paper_id} with combo {param_combo_str_current} already complete.")
                    processed_combos += 1
                    continue

                # Extract current parameters from the configuration grid
                combo = ParamCombo(
                    model=row_combo["model"],
                    temperature=None if pd.isna(row_combo["temperature"]) else float(row_combo["temperature"]),
                    reasoning_effort=None if pd.isna(row_combo["reasoning_effort"]) else str(row_combo["reasoning_effort"]),
                    chain_of_thought=row_combo["chain_of_thought"],
                    note_taking=row_combo["note_taking"]
                )

                # Prepare the setup across the batch
                all_tasks_to_run.append(run_parameter_combo(combo, paper_data, client=client))
                task_metadata.append({
                    "paper_id": paper_id,
                    "param_combo_str": param_combo_str_current,
                    "combo": combo,
                    "paper_data": paper_data
                })

        print(f"Total unique paper-parameter combos to process: {len(all_tasks_to_run)}")

        # Process tasks in batches
        for i in tqdm(range(0, len(all_tasks_to_run), NUM_PARALLEL_BATCH_SIZE), desc="Processing Batches"):
            batch_tasks = all_tasks_to_run[i:i + NUM_PARALLEL_BATCH_SIZE]
            batch_metadata = task_metadata[i:i + NUM_PARALLEL_BATCH_SIZE]

            print(f"\n[RUN BATCH] Processing {len(batch_tasks)} tasks concurrently.", flush=True)

            results = await asyncio.gather(*batch_tasks, return_exceptions=True)

            # Initialize result and log dic for current batch
            batch_results_dicts = []
            batch_log_dicts = []

            # Extract information from each result in the batch
            for j, result in enumerate(results):
                metadata = batch_metadata[j]
                paper_id = metadata["paper_id"]
                param_combo_str = metadata["param_combo_str"]
                combo_obj = metadata["combo"]

                # Initialize result record
                record_for_results = None

                if isinstance(result, Exception):
                    print(f"[ERROR BATCH] Paper {paper_id} with combo {param_combo_str} -> {type(result).__name__}: {result}", flush=True)

                    # Create a dictionary record for results with error status
                    error_record = {
                        "paper_id": paper_id,
                        "model": combo_obj.model,
                        "temperature": combo_obj.temperature,
                        "reasoning_effort": combo_obj.reasoning_effort,
                        "chain_of_thought": combo_obj.chain_of_thought,
                        "note_taking": combo_obj.note_taking,
                        "run_signature": RUN_SIGNATURE
                    }
                    # Populate all llm_notes_X and llm_review_X columns with error messages
                    for review_idx in range(NUM_ITERATIONS_PER_PAPER_PER_COMBO):
                        error_record[f"llm_notes_{review_idx + 1}"] = UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING
                        error_record[f"llm_review_{review_idx + 1}"] = UNKNOWN_ERROR_STATE_PARALLEL_PRCOESSING
                    record_for_results = error_record

                else:
                    # Append result to the record
                    record_for_results = result.to_dict(orient='records')[0]

                # Always append the constructed result record to the batch list
                batch_results_dicts.append(record_for_results)

                # Always add to done_keys to mark this combo as processed/attempted
                done_keys.add((paper_id, param_combo_str))

                # Prepare and append log entry for every processed combo
                new_log_entry = {
                    'paper_id': paper_id,
                    'completed_param_combo': param_combo_str
                }
                batch_log_dicts.append(new_log_entry)

            # After processing all results in the batch, append them at once to JSONL files
            if batch_results_dicts:
                with open(RESULTS_JSON_PATH, 'a') as f:
                    for record in batch_results_dicts:
                        f.write(json.dumps(record) + '\n')
                print(f"Appended {len(batch_results_dicts)} results to {RESULTS_JSON_PATH}")

            if batch_log_dicts:
                with open(LOG_JSON_PATH, 'a') as f:
                    for record in batch_log_dicts:
                        f.write(json.dumps(record) + '\n')
                print(f"Appended {len(batch_log_dicts)} log entries to {LOG_JSON_PATH}")


    await main() # Run the async main function

Processing and saving intermediate results to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_results.jsonl
Logging successful iterations to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_log.jsonl
Loaded 4 completed paper-parameter combo entries from JSON log.
Loaded 4 completed paper-parameter combo entries from JSONL results.
Total already-completed paper-configuration combos: 4
[SKIP] Paper author_0 with combo run=bd301abd5453|model=gpt-5-mini-2025-08-07|temp=None|re=low|cot=|notes=Faithful already complete.
[SKIP] Paper author_1 with combo run=bd301abd5453|model=gpt-5-mini-2025-08-07|temp=None|re=low|cot=|notes=Faithful already complete.
[SKIP] Paper author_0 with combo run=bd301abd5453|model=gpt-5-mini-2025-08-07|temp=None|re=low|cot=|notes=Generic already complete.
[SKIP] Paper author_1 with combo run=bd301abd5453|model=gpt-5-mini-2025-08-07|temp=None|re=low|cot=|notes=Generic already complete.
Total unique p

Processing Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[RUN BATCH] Processing 10 tasks concurrently.
[LLM CALL] paper=author_2 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_3 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_4 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_5 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_6 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_7 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_number=1
[LLM CALL] paper=author_8 | model=gpt-5-mini-2025-08-07, reasoning_effort=low, chain_of_thought=, note_taking=Faithful | review_num

In [27]:
# @title Convert JSONL to Parquet

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_results.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_results.parquet...
Successfully converted and saved results to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_results.parquet.
Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_log.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_log.parquet...
Successfully converted and saved log to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_log.parquet.


In [28]:
# @title Read Parquet files

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df)
        display(llm_results_df.shape)
    except:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}.")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}.")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...
1,author_3,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: Generating Sequences by Learning to [Se...,# Summary Of The Paper\nThis paper proposes Se...,"Paper: ""Generating Sequences by Learning to [S...",# Summary Of The Paper\nThe paper proposes SEL...
2,author_4,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThe paper proposes a u...,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThis paper proposes a ...
3,author_5,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""FROM t-SNE TO UMAP WITH CONTRASTIVE LE...",# Summary Of The Paper\nThis paper provides a ...,"Paper: ""From t-SNE to UMAP with Contrastive Le...",# Summary Of The Paper\nThe paper investigates...
4,author_6,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""Reward Learning with Trees: Methods an...",# Summary Of The Paper\nThis paper studies pre...,Title: Reward Learning with Trees: Methods and...,# Summary Of The Paper\nThe paper studies lear...
5,author_7,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title (anon): Bringing Robotics Taxonomies to ...,# Summary Of The Paper\nThe paper introduces G...,Paper title: Bringing Robotics Taxonomies to C...,# Summary Of The Paper\nThis paper addresses t...
6,author_8,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""ConserWeightive Behavioral Cloning for...",# Summary Of The Paper\nThis paper addresses s...,Title: ConserWeightive Behavioral Cloning for ...,# Summary Of The Paper\nThis paper studies con...
7,author_9,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: Reversible Column Networks (RevCol) — I...,# Summary Of The Paper\nThe paper introduces R...,"Paper: ""Reversible Column Networks (RevCol)"" —...",# Summary Of The Paper\nThe paper proposes Rev...
8,author_2,gpt-5-mini-2025-08-07,NaN,low,,Generic,bd301abd5453,Summary\n- The paper addresses a machine learn...,# Summary Of The Paper\nThe paper proposes a n...,Summary\n- The submission addresses a problem ...,# Summary Of The Paper\nThe submission propose...
9,author_3,gpt-5-mini-2025-08-07,NaN,low,,Generic,bd301abd5453,Summary\n- The paper addresses an important pr...,# Summary Of The Paper\nThe paper addresses im...,Summary\n- The paper addresses a problem in ma...,# Summary Of The Paper\nThe paper proposes a n...


(16, 11)

,paper_id,completed_param_combo
0,author_2,run=bd301abd5453|model=gpt-5-mini-2025-08-07|t...
1,author_3,run=bd301abd5453|model=gpt-5-mini-2025-08-07|t...
2,author_4,run=bd301abd5453|model=gpt-5-mini-2025-08-07|t...
3,author_5,run=bd301abd5453|model=gpt-5-mini-2025-08-07|t...
4,author_6,run=bd301abd5453|model=gpt-5-mini-2025-08-07|t...


(16, 2)